# Leiden sanity check: communities and their lineage over time
Same co-citation slices as `spectral_sanity.ipynb`. Checks, in order:
1. one slice: do the communities have names?
2. is the partition stable across random seeds?
3. does it agree with the spectral picture?
4. timeline of partitions
5. lineage: which communities persist, split, merge
6. a flow picture
7. the export tables the site will consume

In [ ]:
import sys, os, time, numpy as np, pandas as pd, matplotlib.pyplot as plt
import scipy.sparse as sp
ROOT = "."                 # <-- directory that contains lawgraph/
DB   = "graph.duckdb"      # <-- your file
sys.path.insert(0, ROOT)
from lawgraph.store import Store
from lawgraph.lens import select, build, Spectral, Leiden, lineage
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 70)
st = Store(DB, read_only=True); con = st.con

COURT   = "scotus"
STEP    = 5                # years between slices (20 for a quick run, 5 for the animation)
YEARS   = list(range(1900, 2021, STEP))
MODE    = "cocite"
MIN_DEG = 5
RES     = 1.0              # Leiden resolution (modularity). Higher = more, smaller communities.
MIN_SHOW = 50              # only describe communities at least this big

def names(ids):
    df = con.execute("SELECT id, case_name, year(date_filed) AS year FROM node_cluster WHERE id IN (SELECT unnest(?))",
                     [list(map(int, ids))]).df().set_index("id")
    return df.reindex(ids)

def scotus(snap):
    return select(snap, "cluster", where=f"d.dst_id = '{COURT}' AND n.precedential_status = 'Published'",
                  join="JOIN edge_cluster_in_docket_docket cd ON cd.src_id = n.id JOIN edge_docket_filed_in_court d ON d.src_id = cd.dst_id")

def describe(part, G, k=6, min_size=MIN_SHOW):
    """Print each community's size, median year and top-k members by degree."""
    deg = G.degree
    for c in range(part.k):
        m = np.where(part.labels == c)[0]
        if len(m) < min_size: continue
        top = m[np.argsort(-deg[m])[:k]]
        yrs = names(G.ids[m]).year
        print(f"--- community {c}: {len(m):,} cases, median year {int(np.nanmedian(yrs))}")
        for _, r in names(G.ids[top]).iterrows(): print(f"      {str(r.case_name)[:62]}  {r.year}")

def ari(a, b):
    """Adjusted Rand index between two labelings of the same nodes."""
    from scipy.special import comb
    n = len(a); M = sp.csr_matrix((np.ones(n), (a, b))).toarray()
    s = comb(M, 2).sum(); sa = comb(M.sum(1), 2).sum(); sb = comb(M.sum(0), 2).sum()
    e = sa * sb / comb(n, 2); return float((s - e) / ((sa + sb) / 2 - e))

## 1. One slice: do the communities have names?
**Sensible:** 8–25 communities above `MIN_SHOW`, each describable as an area of law from its top cases. A handful of tiny leftovers is normal.

In [ ]:
t0 = time.time()
s = st.at(2000)
G = build(s, ("cluster", "cites", "cluster"), scotus(s), mode=MODE, min_degree=MIN_DEG)
print(f"graph {G.n:,} nodes, {G.W.nnz//2:,} edges   ({time.time()-t0:.1f}s)")
for r in (0.5, 1.0, 2.0):
    t0 = time.time(); p = Leiden(resolution=r)(G)
    sz = p.sizes(); print(f"resolution {r}: {p.k} communities, {int((sz >= MIN_SHOW).sum())} with >= {MIN_SHOW} nodes, "
                          f"largest {sz[:5].tolist()}, quality {p.quality:.3f}   ({time.time()-t0:.1f}s)")
part = Leiden(resolution=RES)(G)
describe(part, G)

## 2. Stability across seeds
**Sensible:** adjusted Rand index between runs ≥ 0.7. Below ~0.5 the partition is not reproducible at this resolution — lower `RES`.

In [ ]:
parts_seed = [Leiden(resolution=RES, seed=sd)(G) for sd in (0, 1, 2)]
for i in range(3):
    for j in range(i + 1, 3):
        print(f"seed {i} vs {j}: ARI = {ari(parts_seed[i].labels, parts_seed[j].labels):.3f}   (k = {parts_seed[i].k}, {parts_seed[j].k})")

## 3. Agreement with the spectral picture
**Sensible:** each arm of the spectral star is (mostly) one colour; communities are contiguous regions, not salt-and-pepper.

In [ ]:
emb = Spectral(k=12)(G)
big = np.where(part.sizes() >= MIN_SHOW)[0]
lab = np.where(np.isin(part.labels, big), part.labels, -1)
fig, ax = plt.subplots(1, 2, figsize=(13, 6))
ax[0].scatter(emb.vecs[:, 0], emb.vecs[:, 1], s=2, alpha=.5, c=lab, cmap="tab20"); ax[0].set_title(f"Leiden communities on spectral coords ({len(big)} shown)")
ax[1].scatter(emb.vecs[:, 2], emb.vecs[:, 3], s=2, alpha=.5, c=lab, cmap="tab20"); ax[1].set_title("same, eigenvectors 3 x 4")
plt.show()
# how pure is each community in spectral terms? mean cosine of members to their community centroid
V = emb.vecs[:, :8] / np.linalg.norm(emb.vecs[:, :8], axis=1, keepdims=True)
rows = []
for c in big:
    m = part.labels == c; cen = V[m].mean(0); cen /= np.linalg.norm(cen)
    rows.append((c, int(m.sum()), round(float((V[m] @ cen).mean()), 3)))
display(pd.DataFrame(rows, columns=["community", "size", "mean cosine to centroid"]).set_index("community"))

## 3b. Interactive 3-D view
Rotate / zoom / pan; hover for case name and year. `DIMS` picks which eigenvectors are the axes (0-based). Needs `pip install plotly`.

In [ ]:
import plotly.graph_objects as go
DIMS = (0, 1, 2)                      # eigenvectors to use as x, y, z
SHOW_LABELS_FOR_TOP = 40              # always-visible text labels for the highest-degree cases
meta = names(G.ids)
hover = [f"{n}<br>{y}  ·  community {c}  ·  degree {int(d)}" for n, y, c, d in zip(meta.case_name, meta.year, part.labels, G.degree)]
X = emb.vecs[:, list(DIMS)]
fig = go.Figure()
for c in big:
    m = part.labels == c
    fig.add_trace(go.Scatter3d(x=X[m, 0], y=X[m, 1], z=X[m, 2], mode="markers", name=f"community {c} ({m.sum():,})",
                               marker=dict(size=2, opacity=0.6), text=[hover[i] for i in np.where(m)[0]], hoverinfo="text"))
top = np.argsort(-G.degree)[:SHOW_LABELS_FOR_TOP]
fig.add_trace(go.Scatter3d(x=X[top, 0], y=X[top, 1], z=X[top, 2], mode="text", text=[str(n)[:30] for n in meta.case_name.values[top]],
                           textfont=dict(size=8), showlegend=False, hoverinfo="skip"))
fig.update_layout(height=800, scene=dict(xaxis_title=f"ev{DIMS[0]+1}", yaxis_title=f"ev{DIMS[1]+1}", zaxis_title=f"ev{DIMS[2]+1}"),
                  legend=dict(itemsizing="constant"), margin=dict(l=0, r=0, t=30, b=0), title=f"{COURT} {MODE}, {G.t}: Leiden communities in spectral space")
fig.show()
# fig.write_html("spectral3d.html")   # standalone file you can open or drop into the site


## 4. Timeline
**Sensible:** community count grows slowly with the graph; quality (modularity) stays in a sensible band (0.3–0.7 for co-citation).

In [ ]:
graphs, parts = {}, {}
for y in YEARS:
    t0 = time.time(); s = st.at(y)
    Gy = build(s, ("cluster", "cites", "cluster"), scotus(s), mode=MODE, min_degree=MIN_DEG)
    if Gy.n < 50: continue
    py = Leiden(resolution=RES)(Gy)
    graphs[y], parts[y] = Gy, py
    sz = py.sizes()
    print(f"{y}: {Gy.n:>7,} nodes  {py.k:>3} communities ({int((sz >= MIN_SHOW).sum())} >= {MIN_SHOW})  "
          f"largest {sz[:4].tolist()}  Q = {py.quality:.3f}   ({time.time()-t0:.1f}s)")

## 5. Lineage
Communities in consecutive slices are linked by shared cases. `overlap` = share of the earlier community that lands in the later one; `jaccard` = symmetric.
**Sensible:** most big communities have one dominant successor (overlap > 0.5); the chains have recognisable names and run for several slices; splits happen where a doctrine actually grew (criminal procedure after 1960, say).

In [ ]:
ys = sorted(parts); plist = [parts[y] for y in ys]
lin = lineage(plist, min_overlap=0.1)
# strongest links between the last two slices
i = len(ys) - 2
tab = pd.DataFrame([(ys[a], ca, ys[b], cb, round(f, 2), round(j, 2)) for a, ca, b, cb, f, j in lin.edges if a == i],
                   columns=["from", "comm", "to", "comm'", "overlap", "jaccard"])
tab = tab[tab.comm.map(lambda c: plist[i].sizes()[c] >= MIN_SHOW)].sort_values(["comm", "overlap"], ascending=[True, False])
display(tab.groupby("comm").head(3).set_index(["from", "comm"]))

In [ ]:
def label_of(i, c, k=3):
    """Name a community by its top-k cases (by degree) in that slice."""
    y = ys[i]; G_, p_ = graphs[y], parts[y]
    m = np.where(p_.labels == c)[0]; top = m[np.argsort(-G_.degree[m])[:k]]
    return " / ".join(str(n)[:28] for n in names(G_.ids[top]).case_name)

chains = [ch for ch in lin.chains(min_jaccard=0.2) if parts[ys[ch[0][0]]].sizes()[ch[0][1]] >= MIN_SHOW]
print(f"{len(chains)} chains of length > 1 (jaccard >= 0.2)\n")
for ch in chains[:25]:
    span = f"{ys[ch[0][0]]}-{ys[ch[-1][0]]}"
    sizes = [int(parts[ys[i]].sizes()[c]) for i, c in ch]
    print(f"{span:>9}  len {len(ch)}  sizes {sizes}")
    print(f"           start: {label_of(*ch[0])}")
    print(f"           end:   {label_of(*ch[-1])}")

In [ ]:
ev = lin.events(min_frac=0.3)
print("splits (a community with >1 significant successor):")
for (i, c), succ in list(ev["splits"].items())[:10]:
    if parts[ys[i]].sizes()[c] < MIN_SHOW: continue
    print(f"  {ys[i]} [{label_of(i, c, 2)}] ->", ", ".join(f"{ys[j]}:{cj} ({f:.2f})" for j, cj, f in succ))
print("\nmerges (a community with >1 significant predecessor):")
for (j, cj), pred in list(ev["merges"].items())[:10]:
    if parts[ys[j]].sizes()[cj] < MIN_SHOW: continue
    print(f"  {ys[j]} [{label_of(j, cj, 2)}] <-", ", ".join(f"{ys[i]}:{ci} ({s:.2f})" for i, ci, s in pred))

## 5b. Animation: spectral communities vs Leiden communities over time
Every slice is rotated into the final orientation (chained Procrustes), so a case only moves when its citation neighbourhood changes.
**Right panel:** colour = Leiden lineage chain (same doctrine = same colour across frames; grey = unlinked).
**Left panel:** spherical k-means on the leading eigenvectors. Each spectral community is matched one-to-one to a Leiden community by Jaccard;
if the match is at least `MATCH` it **borrows that Leiden colour**, otherwise it gets its own colour (a different palette, keyed by its own spectral chain).
So: same colour on both sides = the methods agree on that region; a new colour on the left = spectral sees a grouping Leiden does not.
**Sensible:** arms grow outward as doctrines accumulate; colours persist along arms; agreement (ARI) is moderate-to-high and stable.

In [ ]:
from lawgraph.lens import align_sequence
from lawgraph.lens.community import Partition
from scipy.cluster.vq import kmeans2
from scipy.optimize import linear_sum_assignment
import plotly.graph_objects as go
from plotly.subplots import make_subplots
N_COMM, K_ALIGN = 10, 8
MATCH = 0.5          # Jaccard needed for a spectral community to borrow its matched Leiden community's colour

embs = {y: Spectral(k=12)(graphs[y]) for y in ys}
aligned = dict(zip(ys, align_sequence([embs[y] for y in ys], k=K_ALIGN, reference="last")))

def spectral_partition(y):
    X = embs[y].vecs[:, :N_COMM - 1]; X = X / np.linalg.norm(X, axis=1, keepdims=True)
    _, lab = kmeans2(X, N_COMM, minit="++", seed=0)
    order = np.argsort(-np.bincount(lab, minlength=N_COMM)); remap = np.empty_like(order); remap[order] = np.arange(len(order))
    return Partition(remap[lab], embs[y].ids, str(y))
sparts = {y: spectral_partition(y) for y in ys}

def chain_of_community(partitions):
    """{slice: {community: chain_id}} from lineage over the partitions."""
    lin_ = lineage([partitions[y] for y in ys], min_overlap=0.1)
    cid = {}
    for n, ch in enumerate(lin_.chains(min_jaccard=0.2)):
        for key in ch: cid[key] = n
    return {y: {c: cid.get((i, c), -1) for c in range(partitions[y].k)} for i, y in enumerate(ys)}
lchain_c, schain_c = chain_of_community(parts), chain_of_community(sparts)

def rgb(cmap, i): r, g, b, _ = cmap(i); return f"rgb({int(r*255)},{int(g*255)},{int(b*255)})"
PAL_L = [rgb(plt.cm.tab20, i) for i in range(20)] + [rgb(plt.cm.tab20b, i) for i in range(20)]   # Leiden chains
PAL_S = [rgb(plt.cm.Dark2, i) for i in range(8)] + [rgb(plt.cm.Set1, i) for i in range(9)]        # spectral-only chains
GREY = "#bbbbbb"
def leiden_colour(y, c):
    ch = lchain_c[y][c]; return GREY if ch < 0 else PAL_L[ch % len(PAL_L)]

def match(y):
    """One-to-one Jaccard matching of spectral communities to Leiden communities at slice y (same node set)."""
    S, L = sparts[y], parts[y]
    O = (S.onehot().T @ L.onehot()).toarray()
    J = O / (S.sizes()[:, None] + L.sizes()[None, :] - O + 1e-9)
    r, c = linear_sum_assignment(-J)
    return {int(i): (int(j), float(J[i, j])) for i, j in zip(r, c)}

def spectral_colours(y):
    m = match(y); cols = {}
    for c in range(sparts[y].k):
        j, jac = m.get(c, (None, 0.0))
        if j is not None and jac >= MATCH:
            cols[c] = leiden_colour(y, j)                     # agrees with Leiden: share its colour
        else:
            ch = schain_c[y][c]; cols[c] = GREY if ch < 0 else PAL_S[ch % len(PAL_S)]   # disagrees: own colour
    return [cols[c] for c in sparts[y].labels]
def leiden_colours(y): return [leiden_colour(y, c) for c in parts[y].labels]

# agreement over time
agree = pd.DataFrame({"slice": ys,
    "ARI": [ari(sparts[y].labels, parts[y].labels) for y in ys],
    "matched_above_threshold": [sum(j >= MATCH for _, j in match(y).values()) for y in ys],
    "share_of_cases_sharing_colour": [float(np.mean([a == b for a, b in zip(spectral_colours(y), leiden_colours(y))])) for y in ys]}).set_index("slice")
display(agree.round(3).T)
plt.figure(figsize=(8, 3)); plt.plot(ys, agree.ARI, "o-", label="ARI spectral vs Leiden"); plt.plot(ys, agree.share_of_cases_sharing_colour, "s-", label="share of cases with shared colour")
plt.ylim(0, 1); plt.legend(); plt.title("agreement between the two methods"); plt.show()

def trace(y, cols):
    X = aligned[y]; m = names(graphs[y].ids)
    return go.Scattergl(x=X[:, 0], y=X[:, 1], mode="markers", marker=dict(size=3, color=cols, opacity=.6),
                        text=[f"{n}<br>{yr}" for n, yr in zip(m.case_name, m.year)], hoverinfo="text", showlegend=False)
fig = make_subplots(rows=1, cols=2, subplot_titles=(f"spectral k-means (shares Leiden colour when Jaccard >= {MATCH})", "Leiden chains"),
                    shared_xaxes=True, shared_yaxes=True)
y0 = ys[0]
fig.add_trace(trace(y0, spectral_colours(y0)), row=1, col=1); fig.add_trace(trace(y0, leiden_colours(y0)), row=1, col=2)
fig.frames = [go.Frame(name=str(y), data=[trace(y, spectral_colours(y)), trace(y, leiden_colours(y))], traces=[0, 1]) for y in ys]
allX = np.vstack([aligned[y][:, :2] for y in ys]); pad = 0.05 * (allX.max(0) - allX.min(0))
fig.update_xaxes(range=[allX[:, 0].min() - pad[0], allX[:, 0].max() + pad[0]]); fig.update_yaxes(range=[allX[:, 1].min() - pad[1], allX[:, 1].max() + pad[1]])
fig.update_layout(height=650, width=1300, title=f"{COURT} {MODE}: communities over time, {STEP}-year slices (ev1 x ev2, aligned to {ys[-1]})",
    updatemenus=[dict(type="buttons", x=0, y=1.12, buttons=[
        dict(label="Play", method="animate", args=[None, dict(frame=dict(duration=600, redraw=True), transition=dict(duration=300), fromcurrent=True)]),
        dict(label="Pause", method="animate", args=[[None], dict(frame=dict(duration=0), mode="immediate")])])],
    sliders=[dict(steps=[dict(label=str(y), method="animate", args=[[str(y)], dict(frame=dict(duration=0, redraw=True), mode="immediate")]) for y in ys],
                  x=0.1, len=0.85, currentvalue=dict(prefix="slice: "))])
fig.show()
# fig.write_html("communities_over_time.html")


## 5c. Smooth animation test (video)
Positions computed **yearly** (spectral only, seconds per year), aligned into one orientation; colours = Leiden chain, forward-filled from the
5-year keyframes in section 4. Between consecutive years every case is linearly tweened and new cases fade in; colour changes crossfade.
Black background, one magma shade per Leiden chain, and each community with at least `LABEL_MIN` cases is labelled by its most-cited member at the community's current median position. Writes `smooth_<court>.mp4` (needs ffmpeg) or `.gif`. Runtime: a few minutes for 120 years of SCOTUS.
**Sensible:** arms extend continuously, nothing jumps, a chain split reads as an arm gradually changing colour.

In [ ]:
import matplotlib.animation as anim
from matplotlib import patheffects
POS_STEP, FPS, SEC_PER_YEAR = 1, 20, 0.5          # yearly keyframes, 10 tween frames per year -> 60 s for 120 years
DIMS_V = (0, 1)
LABEL_MIN = 300                                    # label a community when it has at least this many cases at the keyframe
MAX_LABELS = 14
CMAP, BG = plt.cm.magma, "black"
WIDTH, HEIGHT, DPI = 3840, 2160, 200       # 4K; use 1920, 1080, 100 for a quick preview
PT = 5                                     # point size (points^2)
RING = 1.12                                # label ring radius as a fraction of the cloud's max radius
ANG_MIN = np.radians(7)                    # minimum angular gap between labels
SMOOTH_SEC = 1.2                           # time constant for label motion (seconds); larger = slower, calmer
years_pos = list(range(YEARS[0], YEARS[-1] + 1, POS_STEP))

# 1. yearly spectral embeddings (reuse the ones already computed where possible)
t0 = time.time(); pos_embs = {}
for n, y in enumerate(years_pos):
    if y in embs: pos_embs[y] = embs[y]; continue
    s = st.at(y); Gy = build(s, ("cluster", "cites", "cluster"), scotus(s), mode=MODE, min_degree=MIN_DEG)
    if Gy.n < K_ALIGN + 2: continue
    pos_embs[y] = Spectral(k=12)(Gy)
    if n % 10 == 0: print(f"{y}: {Gy.n:,} nodes   ({time.time()-t0:.0f}s)")
years_pos = sorted(pos_embs)
al = dict(zip(years_pos, align_sequence([pos_embs[y] for y in years_pos], k=K_ALIGN, reference="last")))

# 2. one row per case, one position per year (NaN when absent)
all_ids = np.unique(np.concatenate([pos_embs[y].ids for y in years_pos])); N = len(all_ids)
P = np.full((len(years_pos), N, 2), np.nan)
for i, y in enumerate(years_pos):
    P[i, np.searchsorted(all_ids, pos_embs[y].ids)] = al[y][:, list(DIMS_V)]

# 3. colours: one magma sample per Leiden chain (shuffled so neighbouring chain ids differ), forward-filled between keyframes
chains_all = sorted({ch for y in ys for ch in lchain_c[y].values() if ch >= 0})
perm = np.random.default_rng(0).permutation(len(chains_all))
chain_rgb = {ch: CMAP(0.25 + 0.7 * perm[i] / max(len(chains_all) - 1, 1))[:3] for i, ch in enumerate(chains_all)}
GREY3 = np.array([0.35, 0.35, 0.35])
C = np.tile(GREY3, (len(years_pos), N, 1)).astype(float)
keyframe_of = {}
for i, y in enumerate(years_pos):
    ky = max([k for k in ys if k <= y], default=None); keyframe_of[y] = ky
    if ky is None: continue
    idx = np.searchsorted(all_ids, parts[ky].ids)
    C[i, idx] = np.array([chain_rgb.get(lchain_c[ky][c], GREY3) for c in parts[ky].labels])

# 4. labels: per keyframe, each big community -> its top-degree case name + member rows (position = median of members at that year)
labels_at = {}
prev_key = {k: (ys[n - 1] if n else None) for n, k in enumerate(ys)}
for ky in ys:
    G_, p_ = graphs[ky], parts[ky]; rows = []
    for c in range(p_.k):
        m = np.where(p_.labels == c)[0]
        if len(m) < LABEL_MIN: continue
        top = m[np.argmax(G_.degree[m])]
        rows.append((len(m), str(names([G_.ids[top]]).case_name.iloc[0])[:34], np.searchsorted(all_ids, G_.ids[m])))
    labels_at[ky] = sorted(rows, key=lambda r: -r[0])[:MAX_LABELS]

# 5. tween and render
n_tw = max(1, int(FPS * SEC_PER_YEAR))
lo, hi = np.nanmin(P, axis=(0, 1)), np.nanmax(P, axis=(0, 1))
center = np.nanmedian(P[-1], axis=0); R = float(np.nanpercentile(np.linalg.norm(P - center, axis=2), 99.5))   # ignore stray outliers when framing
fig = plt.figure(figsize=(WIDTH / DPI, HEIGHT / DPI), dpi=DPI, facecolor=BG)
ax = fig.add_axes([0, 0, 1, 1]); ax.set_facecolor(BG); ax.set_axis_off(); ax.set_aspect("equal")
half_h = R * RING * 1.15; half_w = half_h * WIDTH / HEIGHT
ax.set_xlim(center[0] - half_w, center[0] + half_w); ax.set_ylim(center[1] - half_h, center[1] + half_h)
sc = ax.scatter([], [], s=PT, linewidths=0)
year_txt = ax.text(0.02, 0.96, "", transform=ax.transAxes, fontsize=26, va="top", color="white", alpha=.9)
ax.text(0.02, 0.03, f"{COURT.upper()} co-citation · Leiden communities · spectral coordinates", transform=ax.transAxes, fontsize=9, color="gray")
pool = [ax.text(0, 0, "", fontsize=8, color="white", va="center", alpha=0,
                path_effects=[patheffects.withStroke(linewidth=2.5, foreground=BG)]) for _ in range(MAX_LABELS)]
lines = [ax.plot([], [], lw=0.6, color="white", alpha=0)[0] for _ in range(MAX_LABELS)]

def place(anchors):
    """Radial label positions on a ring outside the cloud, spread so no two labels are closer than ANG_MIN."""
    d = anchors - center; ang = np.arctan2(d[:, 1], d[:, 0])
    order = np.argsort(ang); a = ang[order]
    for _ in range(20):                                           # relax angular overlaps (circular)
        moved = False
        for k in range(len(a)):
            gap = (a[(k + 1) % len(a)] - a[k]) % (2 * np.pi) if len(a) > 1 else np.inf
            if gap < ANG_MIN:
                shift = (ANG_MIN - gap) / 2; a[k] -= shift; a[(k + 1) % len(a)] += shift; moved = True
        if not moved: break
    out = np.empty_like(anchors); out[order] = center + R * RING * np.c_[np.cos(a), np.sin(a)]
    return out

EASE = 1 - np.exp(-1 / (FPS * SMOOTH_SEC))          # per-frame easing factor
state = {}                                          # label name -> dict(anchor, angle) smoothed over frames

def wrap(a): return (a + np.pi) % (2 * np.pi) - np.pi

def frame(f):
    i, t = divmod(f, n_tw); t /= n_tw
    if i >= len(years_pos) - 1: i, t = len(years_pos) - 2, 1.0
    A, B = P[i], P[i + 1]; both = ~np.isnan(A[:, 0]) & ~np.isnan(B[:, 0]); new = np.isnan(A[:, 0]) & ~np.isnan(B[:, 0])
    pos = np.where(both[:, None], (1 - t) * A + t * B, B)
    alpha = np.where(both, 0.75, np.where(new, 0.75 * t, 0.0))
    col = np.clip((1 - t) * C[i] + t * C[i + 1], 0, 1)
    keep = alpha > 0
    sc.set_offsets(pos[keep]); sc.set_facecolors(np.c_[col[keep], alpha[keep]])
    yr = years_pos[i] + t * (years_pos[i + 1] - years_pos[i]); year_txt.set_text(str(int(round(yr))))
    ky = keyframe_of[years_pos[i]]
    rows = labels_at.get(ky, []) if ky is not None else []
    prev_names = {r[1] for r in labels_at.get(prev_key.get(ky), [])}
    fade_new = float(np.clip((yr - ky) / 0.5, 0, 1)) if ky is not None else 0.0
    # targets for this frame
    targets = {}
    for _, name, idx in rows:
        pp = pos[idx]; pp = pp[~np.isnan(pp[:, 0])]
        if len(pp): targets[name] = (np.median(pp, axis=0), 1.0 if name in prev_names else fade_new)
    # ease each label's anchor toward its target; new labels start at their target
    for name, (anc, _) in targets.items():
        st_ = state.setdefault(name, {"anchor": anc.copy(), "angle": float(np.arctan2(*(anc - center)[::-1]))})
        st_["anchor"] += EASE * (anc - st_["anchor"])
    names_now = list(targets)
    anchors = np.array([state[n]["anchor"] for n in names_now]) if names_now else np.zeros((0, 2))
    spots = place(anchors) if names_now else np.zeros((0, 2))
    for n, sp_ in zip(names_now, spots):                 # ease the ring angle too (handles wrap-around)
        tgt = float(np.arctan2(*(sp_ - center)[::-1]))
        state[n]["angle"] += EASE * wrap(tgt - state[n]["angle"])
    for k, (txt, ln) in enumerate(zip(pool, lines)):
        if k < len(names_now):
            n = names_now[k]; a = state[n]["anchor"]; ang = state[n]["angle"]; fade = targets[n][1]
            sp_ = center + R * RING * np.array([np.cos(ang), np.sin(ang)])
            txt.set_position(sp_); txt.set_text(n); txt.set_ha("left" if sp_[0] >= center[0] else "right"); txt.set_alpha(0.9 * fade)
            ln.set_data([a[0], sp_[0]], [a[1], sp_[1]]); ln.set_alpha(0.35 * fade)
        else:
            txt.set_alpha(0); ln.set_alpha(0)
    return [sc, year_txt, *pool, *lines]
n_frames = (len(years_pos) - 1) * n_tw
mov = anim.FuncAnimation(fig, frame, frames=n_frames, interval=1000 / FPS, blit=True)
out = f"smooth_{COURT}_{HEIGHT}p"
t0 = time.time()
try:
    mov.save(out + ".mp4", writer=anim.FFMpegWriter(fps=FPS, bitrate=12000 if HEIGHT >= 2000 else 4000), dpi=DPI, savefig_kwargs=dict(facecolor=BG)); print(f"wrote {out}.mp4   ({time.time()-t0:.0f}s)")
except Exception as e:
    print("no ffmpeg (" + str(e)[:60] + "), writing gif instead"); mov.save(out + ".gif", writer=anim.PillowWriter(fps=FPS), dpi=DPI, savefig_kwargs=dict(facecolor=BG)); print("wrote", out + ".gif")
plt.close(fig)


## 5d. Summaries for the labelled cases
`case_summaries.csv` holds a two-sentence summary (checked against Justia / U.S. Reports) for the cases that become labels. This cell matches them to the labels the animation used and lists any label without a summary, so you know what to add.

In [ ]:
summ = pd.read_csv("case_summaries.csv")
label_names = sorted({name for ky in ys for _, name, _ in labels_at.get(ky, [])})
def find(label):
    for _, r in summ.iterrows():
        if r.name_key.lower() in label.lower() or label.lower()[:20] in r.case_name.lower(): return r
    return None
hits = {lbl: find(lbl) for lbl in label_names}
print(f"{sum(h is not None for h in hits.values())} of {len(label_names)} label cases have a summary\n")
for lbl, r in hits.items():
    if r is None: print(f"MISSING  {lbl}")
print()
for lbl, r in hits.items():
    if r is not None: print(f"{r.case_name} ({r.year}, {r.citation})\n   {r.summary}\n")


## 5e. Export for the interactive viewer
Writes `viewer/data/`: quantised yearly 3-D positions in decade chunks, chain ids per year, a case index, label anchors, summaries, and top-10 co-citation neighbours per case per decade as one small binary each. Needs the arrays from 5c (`years_pos`, `all_ids`, `al`, `labels_at`, `chain_rgb`, `lchain_c`). Then serve `viewer/` as static files.

In [ ]:
from lawgraph.viewer_export import export_viewer, cocitation_neighbors
VIEWER_OUT = "viewer/data"

# 3-D aligned positions per year (NaN where absent)
Tn = len(years_pos); pos3 = np.full((Tn, N, 3), np.nan, dtype=np.float32)
for i, y in enumerate(years_pos):
    pos3[i, np.searchsorted(all_ids, pos_embs[y].ids)] = al[y][:, :3]

# chain id per case per year, forward-filled from the Leiden keyframes
chain_year = np.full((Tn, N), -1, dtype=np.int32)
for i, y in enumerate(years_pos):
    ky = keyframe_of[y]
    if ky is None: continue
    idx = np.searchsorted(all_ids, parts[ky].ids)
    chain_year[i, idx] = [lchain_c[ky][c] for c in parts[ky].labels]

# label anchors per keyframe: (name, x, y, z) at the community's median 3-D position, plus chain id for the legend
labels_year = {}
for ky in ys:
    i = years_pos.index(ky) if ky in years_pos else None
    if i is None: continue
    rows = []
    for size, name, idx in labels_at[ky]:
        pp = pos3[i, idx]; pp = pp[~np.isnan(pp[:, 0])]
        comm = parts[ky].labels[np.searchsorted(parts[ky].ids, all_ids[idx[0]])]      # community of these members -> its chain
        if len(pp): rows.append((name, *np.median(pp, axis=0), lchain_c[ky][int(comm)]))
    labels_year[ky] = rows

meta_all = names(all_ids)
decades = list(range((years_pos[0] // 10) * 10, years_pos[-1] + 1, 10))
t0 = time.time(); nbrs = cocitation_neighbors(lambda y: st.at(y), decades, all_ids, k=10); print(f"neighbours: {len(nbrs):,} rows ({time.time()-t0:.0f}s)")
summ = pd.read_csv("case_summaries.csv") if os.path.exists("case_summaries.csv") else None
export_viewer(VIEWER_OUT, years_pos, all_ids, pos3, chain_year, chain_rgb, meta_all, labels_year, neighbors=nbrs, summaries=summ)


## 6. Flow picture
Bars = communities per slice (height ∝ size), lines = shared cases between consecutive slices (width ∝ count). The site's Sankey is this with labels.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
pos = {}
for xi, y in enumerate(ys):
    sz = parts[y].sizes(); y0 = 0
    for c, n in enumerate(sz):
        if n < MIN_SHOW: continue
        ax.add_patch(plt.Rectangle((xi - 0.08, y0), 0.16, n, color=plt.cm.tab20(c % 20)))
        pos[(xi, c)] = (y0, y0 + n); y0 += n * 1.08
for a, ca, b, cb, f, j in lin.edges:
    if (a, ca) in pos and (b, cb) in pos and j >= 0.05:
        (l0, l1), (r0, r1) = pos[(a, ca)], pos[(b, cb)]
        ax.plot([a + 0.08, b - 0.08], [(l0 + l1) / 2, (r0 + r1) / 2], lw=max(0.3, 8 * j), alpha=.35, color="gray")
ax.set_xticks(range(len(ys))); ax.set_xticklabels(ys); ax.set_ylabel("cases"); ax.set_xlim(-0.5, len(ys) - 0.5)
ax.set_ylim(0, max(e for s, e in pos.values()) * 1.02); ax.set_title(f"community flow, {COURT} {MODE}, resolution {RES}"); plt.show()

## 7. Export tables
What the site needs: one row per (slice, case) with its community and chain, and one row per lineage edge. Set `EXPORT = True` to write Parquet.

In [ ]:
EXPORT = False
OUT = "out"
chain_id = {}                                        # (slice_index, community) -> chain number
for n, ch in enumerate(lin.chains(min_jaccard=0.2)):
    for key in ch: chain_id[key] = n
nodes = pd.concat([pd.DataFrame({"slice": y, "cluster_id": parts[y].ids, "community": parts[y].labels,
                                 "chain": [chain_id.get((i, c), -1) for c in parts[y].labels],
                                 **({"x": aligned[y][:, 0], "y": aligned[y][:, 1], "z": aligned[y][:, 2],
                                     "spectral_community": sparts[y].labels, "spectral_chain": [schain_c[y][c] for c in sparts[y].labels],
                                     "spectral_colour": spectral_colours(y), "leiden_colour": leiden_colours(y)} if "aligned" in globals() else {})})
                   for i, y in enumerate(ys)], ignore_index=True)
edges = pd.DataFrame([(ys[a], ca, ys[b], cb, int(lin.overlaps[a][ca, cb]), round(f, 4), round(j, 4))
                      for a, ca, b, cb, f, j in lin.edges],
                     columns=["from_slice", "from_comm", "to_slice", "to_comm", "shared", "overlap", "jaccard"])
labels = pd.DataFrame([(ys[i], c, int(parts[ys[i]].sizes()[c]), label_of(i, c, 3))
                       for i in range(len(ys)) for c in range(parts[ys[i]].k) if parts[ys[i]].sizes()[c] >= MIN_SHOW],
                      columns=["slice", "community", "size", "label"])
print(nodes.shape, edges.shape, labels.shape); display(labels.head(12))
if EXPORT:
    import os; os.makedirs(OUT, exist_ok=True)
    nodes.to_parquet(f"{OUT}/leiden_nodes_{COURT}.parquet", index=False)
    edges.to_parquet(f"{OUT}/leiden_lineage_{COURT}.parquet", index=False)
    labels.to_parquet(f"{OUT}/leiden_labels_{COURT}.parquet", index=False)
    print("written to", OUT)